In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. knowledge base 구성을 위한 데이터 생성

In [7]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법_with_table.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 6.02 s
Wall time: 6.25 s


180

In [3]:
documents[43]

Document(metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억5,000만원 이하\n\n1,536만원 + (8,800만원 초과하는 금액의 35퍼센트)\n\n1억5,000만원 초과 3억원 이하\n\n3,706만원 + (1억5,000만원 

In [8]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True) #커널을 시작하지 않고, .env덮어쓰기
embedding=UpstageEmbeddings(
            model='embedding-query'
)

In [9]:
len(embedding.embed_query('소득세법'))

4096

In [10]:
#파인콘의 인덱스 생성하기=>없을때 크리에이트create해야 함
import time
from pinecone import Pinecone, ServerlessSpec

#0. 파인콘객체생성
pc=Pinecone()#api_key=os.getenv('PINECONE_API_KEY')

#1. 인덱스 설정
index_name='tax-index-table'
dimension=len(embedding.embed_query("차원 확인용")) #5개까지만 가능하므로
              
#2. 인덱스 생성(해당인덱스가 없을 경우)
existing_indexes=[index.name for index in pc.list_indexes()]
print(index_name not in existing_indexes)

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric='cosine', #유사도 측정방식
        spec=ServerlessSpec(
            cloud='aws',  #무료플랜은 aws
            region='us-east-1' #무표플랜은 us-east-1만 가능
        )
    )
    #인덱스가 준비될 때까지 ㅣ대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) #1초 대기
    print(f'인덱스 {index_name}을 생성하였습니다')
else:
    print(f'인덱스 {index_name}이 이미 존재합니다')

True
인덱스 tax-index-table을 생성하였습니다


In [5]:
# 3. 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['tax-index-openai', 'tax-index-table', 'tax-index']
True
{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [13]:
%%time
# # 4. 문서를 임베딩해서 클라우드 vector DB(index)에 저장
#처음한번
from pinecone import Pinecone
pc=Pinecone()
from langchain_pinecone import PineconeVectorStore
# database=PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )

#업로드한 벡터 DB를 load할 때
database=PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 0 ns
Wall time: 11.5 ms


In [14]:
database.similarity_search("연봉 5천만원인 직장인 소득세는?")

[Document(id='e91bdde2-a232-405b-89e1-eb252255ae4a', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억5,000만원 이하\n\n1,536만원 + (8,800만원 초과하는 금액의 35퍼센트)\n\n

# 2. 답변 생성 전 retieval 확인

In [18]:
query="연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs=database.similarity_search(query,k=3) #기본 k값은 4

In [19]:
retrieved_docs[0]

Document(id='e91bdde2-a232-405b-89e1-eb252255ae4a', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억5,000만원 이하\n\n1,536만원 + (8,800만원 초과하는 금액의 35퍼센트)\n\n1

In [20]:
retrieved_docs[1]

Document(id='59e4c137-9856-4aa2-bfc2-b5f85d82d86c', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='③ 일용근로자의 근로소득에 대해서 제134조제3항에 따른 원천징수를 하는 경우에는 해당 근로소득에 대한 산출세액의 100분의 55에 해당하는 금액을 그 산출세액에서 공제한다.<개정 2014. 1. 1.>\n\n[전문개정 2012. 1. 1.]\n\n\n\n제59조의2(자녀세액공제) ①종합소득이 있는 거주자의 기본공제대상자에 해당하는 자녀(입양자 및 위탁아동을 포함하며, 이하 이 조에서 “공제대상자녀”라 한다) 및 손자녀로서 13세 이상의 사람에 대해서는 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다. <개정 2015. 5. 13., 2017. 12. 19., 2018. 12. 31., 2019. 12. 31., 2022. 12. 31., 2023. 12. 31., 2024. 12. 31., 2026. 4. 21.>\n\n1. 1명인 경우: 연 25만원\n\n2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>\n\n1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경우: 연 30만원\n\n2. 출산하거나 입양 신고한 공제대상자녀가 둘째인 경우: 연 50만원\n\n3. 출산하거나 입양 신고한 공제대상자녀가 셋째 이상인 경우: 연 70만원\n\n④ 제1항 및 제3항에 따른 공제를 “자녀세액공제”라 한다.<신설 2015. 5. 13., 2017. 12. 19.>\n\n[본조신설 2014. 1. 1.]\n\n[종전 제59조의2는 제59조의

In [21]:
retrieved_docs[2]

Document(id='ffac513d-1b04-4329-9a77-b315cc5f1d3f', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제58조(재해손실세액공제) ① 사업자가 해당 과세기간에 천재지변이나 그 밖의 재해(이하 “재해”라 한다)로 대통령령으로 정하는 자산총액(이하 이 항에서 “자산총액”이라 한다)의 100분의 20 이상에 해당하는 자산을 상실하여 납세가 곤란하다고 인정되는 경우에는 다음 각 호의 소득세액(사업소득에 대한 소득세액을 말한다. 이하 이 조에서 같다)에 그 상실된 가액이 상실 전의 자산총액에서 차지하는 비율(이하 이 조에서 “자산상실비율”이라 한다)을 곱하여 계산한 금액(상실된 자산의 가액을 한도로 한다)을 그 세액에서 공제한다. 이 경우 자산의 가액에는 토지의 가액을 포함하지 아니한다. <개정 2020. 12. 29.>\n\n1. 재해 발생일 현재 부과되지 아니한 소득세와 부과된 소득세로서 미납된 소득세액\n\n2. 재해 발생일이 속하는 과세기간의 소득에 대한 소득세액\n\n② 제1항의 경우에 제56조, 제56조의2, 제57조 및 제57조의2에 따라 공제할 세액이 있을 때에는 이를 공제한 후의 세액을 소득세액으로 하여 제1항을 적용한다.<개정 2022. 12. 31.>\n\n③ 제1항에 따른 공제를 “재해손실세액공제”라 한다.\n\n④ 재해손실세액공제를 적용받으려는 자는 대통령령으로 정하는 바에 따라 관할 세무서장에게 신청할 수 있다.\n\n⑤ 관할 세무서장이 제4항의 신청을 받았을 때에는 그 공제할 세액을 결정하여 신청인에게 알려야 한다.\n\n⑥ 제4항의 신청이 없는 경우에도 제1항을 적용한다.\n\n⑦ 집단적으로 재해가 발생한 경우에는 대통령령으로 정하는 바에 따라 관할 세무서장이 조사결정한 자산상실비율에 따라 제1항을 적용한다.\n\n⑧ 재해손실세액공제에 관하여 필요한 사항은 대통령령으로 정한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\

In [23]:
retriever=database.as_retriever(
        search_type='similarity',
        search_kwargs={'k':2}
)
retrieved_docs=retriever.invoke("연봉 5천만원인 직장인의 소득세는 얼마인가요?")
retrieved_docs[0]

Document(id='e91bdde2-a232-405b-89e1-eb252255ae4a', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억5,000만원 이하\n\n1,536만원 + (8,800만원 초과하는 금액의 35퍼센트)\n\n1

In [25]:
retrieved_doc="\n--\n".join([doc.page_content for doc in retrieved_docs])
#print(retrieved_doc)

제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>

[전문개정 2009. 12. 31.]

[제목개정 2014. 1. 1.]



제4절 세액의 계산 <개정 2009. 12. 31.>



제1관 세율 <개정 2009. 12. 31.>



제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>











종합소득 과세표준

세율

1,400만원 이하

과세표준의 6퍼센트

1,400만원 초과 5,000만원 이하

84만원 + (1,400만원 초과하는 금액의 15퍼센트)

5,000만원 초과 8,800만원 이하

624만원 + (5,000만원 초과하는 금액의 24퍼센트)

8,800만원 초과 1억5,000만원 이하

1,536만원 + (8,800만원 초과하는 금액의 35퍼센트)

1억5,000만원 초과 3억원 이하

3,706만원 + (1억5,000만원 초과하는 금액의 38퍼센트)

3억원 초과 5억원 이하

9,406만원 + (3억원 초과하는 금액의 40퍼센트)

5억원  초과 10억원 이하

1억7,406원 + (5억원을 초과하는 금액의 42퍼센트)

10억원 초과

3억8

In [26]:
'55조' in retrieved_doc

True

# 3. 답변 생성

In [31]:
from openai import OpenAI
client=OpenAI()
query="연봉 5천만원인 직장인의 소득세는 얼마인가요"
response=client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role':'system','content':'당신은 한국 최고의 소득세법 전문가입니다'},
        {
            'role':'user',
            'content':f"""-[context]를 참고로 해서 사용자의 질문에 10줄 이내로 답변해 주세요.
            -[context]:{retrieved_doc}
            -질문:{query}"""
        }
    ],
    temperature=0.2
)

In [32]:
result=response.choices[0].message.content
print(result)

연봉 5천만원인 직장인의 소득공제는 여러 요소에 따라 달라질 수 있지만, 기본적으로 적용되는 소득공제 항목은 다음과 같습니다.

1. **기본공제**: 본인과 부양가족에 대한 기본공제가 적용됩니다. 본인 150만원, 부양가족 1인당 150만원이 기본공제로 인정됩니다.
2. **근로소득공제**: 연봉에 따라 자동으로 적용되는 근로소득공제가 있습니다. 5천만원의 경우 약 1,200만원 정도가 공제됩니다.
3. **기타 공제**: 의료비, 교육비, 기부금 등 추가적인 공제가 있을 수 있습니다.

따라서, 기본공제와 근로소득공제를 합산하면 대략 1,350만원 이상의 소득공제를 받을 수 있습니다. 정확한 금액은 개인의 상황에 따라 다를 수 있습니다.


In [33]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model='gpt-4.1-mini')

In [34]:
from langchain_upstage import ChatUpstage
llm=ChatUpstage(
    model='solar-pro4',
    reasoning_effort='low' #답변은 느리지만 더 깊게 추론(low,medium,high)
)

In [36]:
prompt=f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다.
- [context]를 참고하여 사용자의 질문에 답해주세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문:{query}"""

In [38]:
ai_message=llm.invoke(prompt)
print(ai_message.content)

제공된 법령 텍스트와 소득세 전문가로서의 지식을 바탕으로 답변드리겠습니다.

먼저, 질문하신 **"연봉 5천만원 직장인의 소득공제"**에 대해 명확히 구분해야 할 점이 있습니다. 제공해주신 법령 텍스트에는 근로소득자의 기본적인 **소득공제**(과세표준을 줄이는 금액)에 대한 구체적인 산식은 포함되어 있지 않으며, 주로 **세액공제**(산출세액에서 직접 차감하는 금액) 및 **세율**에 대한 규정이 명시되어 있습니다.

따라서 제공된 컨텍스트(제55조, 제59조의2, 제59조의3 등)를 참조하여 답변은 다음과 같이 구분하여 설명드립니다.

### 1. 소득공제 (Income Deduction) 관련
제공된 텍스트(제54조의2, 제55조, 제59조의2, 제59조의3 등)에는 직장인의 급여에서 기본적으로 공제되는 **근로소득공제**나 **인적공제**(기본공제 등)의 구체적인 금액이 명시되어 있지 않습니다. 이는 일반적으로 소득세법 시행령 또는 기본 규정에서 정하는 사항입니다.

전문가로서의 추정치를 덧붙이자면, 연봉 5천만원인 직장인의 경우 다음과 같은 소득공제가 일반적으로 적용됩니다.
*   **근로소득공제**: 총급여액에 따라 필수적으로 적용되며, 5천만원 기준 약 1,095만원 정도로 계산됩니다 (법령상 구간별 공식에 따름).
*   **인적공제**: 본인 및 부양가족(자녀 등) 1인당 연 150만원.
*   **기타 소득공제**: 신용카드 사용금액, 보험료, 의료비 등 추가적인 공제 항목이 있을 수 있습니다.

이 공제들을 차감한 후의 금액이 **과세표준**이 되며, 이 과세표준에 제55조의 세율이 적용됩니다.

### 2. 세액공제 (Tax Credit) 관련 (제공된 컨텍스트 기준)
제공된 텍스트에는 과세표준을 줄이는 '소득공제'가 아니라, 계산된 세금(산출세액)에서 직접 차감하는 **세액공제** 항목들이 명시되어 있습니다. 연봉 5천만원 직장인에게 적용 가능한 주요 항목은 다음과 같습니다.

*   **연금계좌세액공제 (제59조의3)**:
    *  

In [47]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage, UpstageEmbeddings
# from langchain_chroma import Chroma
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 1. 임베딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')
embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')
                  
# 2. vector DB load
#pc=Pinecone()                  
index_name='tax-index-table'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 3. Retriever 생성 - search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':2}) # 유사도가 높은 top-2추출

# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

# 6.RAG랭체인 구성(lcel방식)

In [48]:
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result=rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

제공된 문맥은 제55조 세율표를 통해 과세표준 구간별 산출세액 계산 방식만 명시하고 있으며, 근로소득공제·인적공제·연금보험료공제 등 실제 과세표준을 결정하는 주요 소득공제 항목과 각종 세액공제에 대한 정보가 없습니다. 따라서 연봉 5천만원이라는 총급여액만으로는 정확한 소득세액을 산출할 수 없습니다. 정확한 세금을 알기 위해서는 총 급여액에서 근로소득공제를 뺀 근로소득금액을 기준으로 추가적인 소득공제와 세액공제를 적용하여 과세표준과 최종 산출세액을 계산해야 합니다.
